Import Libraries

In [9]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from dotenv import load_dotenv
from langchain.agents import create_agent
import sqlite3

load_dotenv()
print("Imports Done!!!")

Imports Done!!!


Initialize Claude

In [3]:
llm = ChatAnthropic(
    model = "claude-sonnet-4-6",
    temperature = 0
)
print("Claude Ready!!!")

Claude Ready!!!


Database Setup

In [5]:
def query_db(sql: str) -> list:
    conn = sqlite3.connect('business.db')
    cursor = conn.cursor()
    cursor.execute(sql)
    results = cursor.fetchall()
    conn.close()
    return results

print("Database helper ready!")
print(f"Test - customers count: {query_db('SELECT COUNT(*) FROM customers')[0][0]}")

Database helper ready!
Test - customers count: 10


Define SQL Tools

In [6]:
@tool
def run_sql_query(sql: str) -> str:
    """Executes a SQL SELECT query on the business database and returns results.
    Database has two tables:
    - customers: id, name, email, city, state, age, joined_date
    - orders: id, customer_id, product, amount, quantity, order_date, status
    Write valid SQLite SELECT queries only."""
    try:
        # Safety check - only allow SELECT queries
        if not sql.strip().upper().startswith('SELECT'):
            return "Error: Only SELECT queries are allowed."
        
        results = query_db(sql)
        
        if not results:
            return "No results found for that query."
        
        return f"Query returned {len(results)} rows:\n" + \
               "\n".join([str(row) for row in results])
    except Exception as e:
        return f"SQL Error: {str(e)}"

@tool
def get_schema() -> str:
    """Returns the database schema - table names, columns and data types.
    Always call this first before writing any SQL query."""
    try:
        schema = []
        
        # Get customers schema
        results = query_db("PRAGMA table_info(customers)")
        schema.append("Table: customers")
        schema.append("Columns: " + ", ".join([f"{r[1]} ({r[2]})" for r in results]))
        
        # Get orders schema
        results = query_db("PRAGMA table_info(orders)")
        schema.append("\nTable: orders")
        schema.append("Columns: " + ", ".join([f"{r[1]} ({r[2]})" for r in results]))
        
        # Get row counts
        customers_count = query_db("SELECT COUNT(*) FROM customers")[0][0]
        orders_count = query_db("SELECT COUNT(*) FROM orders")[0][0]
        schema.append(f"\nRecord counts:")
        schema.append(f"  customers: {customers_count} rows")
        schema.append(f"  orders: {orders_count} rows")
        
        return "\n".join(schema)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_sample_data(table_name: str) -> str:
    """Returns 3 sample rows from a table to understand the data format.
    Valid table names: customers, orders"""
    try:
        if table_name not in ['customers', 'orders']:
            return "Error: Valid tables are 'customers' and 'orders' only."
        
        results = query_db(f"SELECT * FROM {table_name} LIMIT 3")
        return f"Sample data from {table_name}:\n" + \
               "\n".join([str(row) for row in results])
    except Exception as e:
        return f"Error: {str(e)}"

tools = [get_schema, get_sample_data, run_sql_query]
llm_with_tools = llm.bind_tools(tools)
print(f"SQL Tools ready: {[t.name for t in tools]}")

SQL Tools ready: ['get_schema', 'get_sample_data', 'run_sql_query']


ReAct Agent

In [11]:
system_prompt = """You are a helpful SQL assistant with access to a business database.
Think step by step before taking any action.
Always check the schema first before writing queries.
Show your reasoning at each step."""

agent = create_agent(
    model=llm,
    tools=[get_schema, get_sample_data, run_sql_query],
    system_prompt=system_prompt
)

print("ReAct SQL Agent ready!!!")

ReAct SQL Agent ready!!!


Invoke the Agent

In [12]:
result = agent.invoke({
    "messages": [HumanMessage(content="How many order are from the State of Texas")]
})

print("\n🤖 Final Answer:")
print(result["messages"][-1].content)


🤖 Final Answer:
There are **7 orders** from the state of **Texas**. 🤠

Here's a quick summary of how I got there:
- **Joined** the `orders` table with the `customers` table using `customer_id`.
- **Filtered** for customers where `state = 'Texas'`.
- **Counted** the matching orders.

Would you like a further breakdown, such as orders by city, customer, or status within Texas?
